In [ ]:
import adaptive_latents as al
import numpy as np
import scipy
import matplotlib.pyplot as plt
from tqdm.autonotebook import  tqdm

from adaptive_latents import ArrayWithTime

from IPython import display
import time
from functools import lru_cache
rng = np.random.default_rng()

In [ ]:
d = al.datasets.Odoherty21Dataset()

In [ ]:
def rbf_kernel(x, y, lengthscale=10):
    return np.exp(-0.5 * np.linalg.norm(x-y)**2 / lengthscale**2)

In [ ]:
dictionary = [d.neural_data[0]]
K = np.array([[rbf_kernel(dictionary[0], dictionary[0])]])

threshold = .01
# Approximate Linear Dependence as a Design Method for Kernel Prototype-Based Classifiers, Coelho & Barreto

for row in tqdm(d.neural_data[1:]):
    k_others = np.array([rbf_kernel(row, d) for d in dictionary])
    k_self = rbf_kernel(row,row)
    ald = k_self - k_others.T @ np.linalg.pinv(K) @ k_others
    print(ald)
    if ald > threshold:
        dictionary.append(row)
        K = np.block([[K, k_others[:,None]], [k_others[None,:], k_self]])


In [ ]:
K.shape

In [ ]:
@lru_cache
def f(r, data=d.neural_data):
    seen = []
    hits = []

    for row in data:
        for i, s in enumerate(seen):
            if np.linalg.norm(row - s) < r:
                if i != 0:
                    seen.insert(0, seen.pop(i))
                    hits.insert(0, hits.pop(i))
                hits[0].append(row.t)
                break
        else:
            seen.insert(0, row)
            hits.insert(0, [row.t])

    seen, hits = zip(*sorted(zip(seen,hits), key=lambda x: -len(x[1])))
    return ArrayWithTime.from_list(seen), hits

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10,5))

xs = np.linspace(2,10,20)[::-1]
ys = np.nan * xs

for i in range(len(xs)):
    seen, hits = f(xs[i])
    n_hits = np.array([len(h) for h in hits])

    ax = axs[0]
    ys[i] = (n_hits>0).sum()
    ax.clear()
    ax.plot(xs, ys, '-o')
    ax.set_xlabel('r (neural distance)')
    ax.set_ylabel('number of unique vectors')
    ax.semilogy()
    ax.grid()

    ax = axs[1]
    ax.clear()
    ax.hist(n_hits[n_hits > 0], bins=100)
    ax.semilogy()

    display.clear_output()
    display.display(fig)

In [ ]:
f(xs[10])[0].slice(slice(None,1000)).shape

In [ ]:
basis = d.neural_data[rng.choice(len(d.neural_data), size=1000, replace=False)]
ny_d = scipy.spatial.distance_matrix(d.neural_data, basis).min(axis=1)


basis = f(xs[10])[0].slice(slice(-1000,None))
dynamic_d = scipy.spatial.distance_matrix(d.neural_data, basis).min(axis=1)

bins = np.linspace(0,8,100)
plt.hist(ny_d, bins=bins, alpha=0.5)
plt.hist(dynamic_d, bins=bins, alpha=0.5);
